In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import gc

In [ ]:
paths = [
    '/kaggle/input/notebooks/fati22/embedding-images-phase1/chunk_1.parquet',
    '/kaggle/input/notebooks/fati22/embedding-images-phase1/chunk_2.parquet',
    #'/kaggle/input/notebooks/fati22/embedding-images-phase2/chunk_3.parquet',
    #'/kaggle/input/notebooks/fati22/embedding-images-phase2/chunk_4.parquet'
]

model = SentenceTransformer('BAAI/bge-m3').cuda()

In [ ]:
for i, p in enumerate(paths):
    df = pd.read_parquet(p)
    titles = df['Judul'].tolist()
    ids = df['ID_Product'].values
    img_emb = df['Embedding'].values
    
    del df
    gc.collect()
    with torch.no_grad():
        judul_vectors = model.encode(titles, batch_size=256, show_progress_bar=True, convert_to_numpy=True)
    df_final = pd.DataFrame({
        'ID_Product': ids,
        'Image_Embedding': img_emb,
        'Judul_Embedding': list(judul_vectors)
    })
    output_name = f"final_chunk_{i+1}.parquet"
    df_final.to_parquet(output_name, engine='pyarrow', compression='snappy')
    
    del df_final, titles, ids, img_emb, judul_vectors
    gc.collect()
    torch.cuda.empty_cache()